# 話者分離つき文字起こし v2 (Colab Pro GPU用、長時間自動対応、一括処理)

対象: Googleレコーダー由来m4a (30分-2時間、中日混じり)。長いものは自動で30分ずつ切って起こし、分離だけ一括で行う。手で割る必要なし。

構成: transformers版whisper turbo (GPU) + sherpa-onnx分離 (HFトークン不要)。

なぜ自動分割か: ファイルサイズ (2時間で20MB) は問題ではない。敵はColabのセッション切断・メモリ・誤差蓄積・中断時のやり直し。チャンク毎に保存するので落ちても続きからいける。

使い方: to_colabに複数置いて上から順に実行。済みは自動スキップ、REDO=Trueで再実行。成功分はoldへ移動。終わったら合図セルで確認。

In [ ]:
import os
_ncpu = os.cpu_count() or 4
os.environ.setdefault("OMP_NUM_THREADS", str(_ncpu))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_ncpu))
os.environ.setdefault("MKL_NUM_THREADS", str(_ncpu))
print(f"OMP_NUM_THREADS={os.environ['OMP_NUM_THREADS']} cpu={_ncpu}")
import torch
if not torch.cuda.is_available():
    raise SystemExit("GPUなしのため停止。ランタイム→タイプ変更でGPUにすること")
print(torch.cuda.get_device_name(0))
!pip install -q transformers sherpa-onnx soundfile numpy

In [ ]:
from google.colab import files
from pathlib import Path
import json as _json
import shutil as _shutil

# 伝書鳩回避: Drive直結。TrueならDriveの入出力を使い、上げ下げ不要
USE_DRIVE = True
DRIVE_BASE = "/content/drive/MyDrive/フロニシス/Gdrive_code"  # Kaiの窓。変える時はここだけ
TO_COLAB_DIR = DRIVE_BASE + "/to_colab"  # 入力置き場
DRIVE_OUTDIR = DRIVE_BASE + "/from_colab"  # 出力先。同期フォルダに合わせる
OLD_DIR = TO_COLAB_DIR + "/old"  # 成功分移動先。無ければ作る
REDO = False  # Trueで出力済みも再実行。普段はFalseのまま

# 設定 (必要なら変える)
LANGUAGE = "ja"  # auto/ja/zh。翻訳はしない (turboはtranscribe専用)
NUM_SPEAKERS = 0  # 人数が分かれば固定 (37分裂の特効薬)。不明なら0
CLUSTER_THRESHOLD = 0.5  # 小さいほど話者数が増える
CHUNK_MIN = 30  # 起こしの分割単位 (分)。変える必要なし
OVERLAP_S = 60  # チャンク先頭の重ね代 (秒)。後チャンク側を捨てて結合
BATCH_SIZE = 16  # VRAM余裕機は32に上げる
MODELS_DIR = DRIVE_BASE + "/models" if USE_DRIVE else "/content/models"  # sherpa分離モデルとHFキャッシュの常駐先。セル間グローバルで受け渡す
HF_HOME_DIR = MODELS_DIR + "/hf_hub"  # HFキャッシュ常駐先。初回のみ取得、2回目以降は再利用
SEG_MODEL_INT8 = MODELS_DIR + "/sherpa-onnx-pyannote-segmentation-3-0/model.int8.onnx"  # セル間受け渡し用
SEG_MODEL_FP32 = MODELS_DIR + "/sherpa-onnx-pyannote-segmentation-3-0/model.onnx"  # int8不在時の退避
EMB_MODEL_PATH = MODELS_DIR + "/nemo_en_titanet_small.onnx"  # セル間受け渡し用
import os as _os2
_os2.environ.setdefault("HF_HOME", HF_HOME_DIR)
_os2.environ.setdefault("HUGGINGFACE_HUB_CACHE", HF_HOME_DIR)
_os2.environ.setdefault("TRANSFORMERS_CACHE", HF_HOME_DIR)
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
Path(HF_HOME_DIR).mkdir(parents=True, exist_ok=True)

OK, FAIL, SKIPPED, FILES = [], [], [], []

def _out_key_for(src_str):
    # Hayato指摘3: 同stem異拡張子は別物扱い。キーに拡張子を含める
    pp = Path(src_str)
    ext = pp.suffix.lower().lstrip(".") or "noext"
    return f"{pp.stem}__{ext}"

def _is_valid_output(outbase, key):
    # Hayato指摘2: json存在だけでなく読込可・segments非空を確認。不正は再処理
    fp = Path(outbase) / (key + ".json")
    try:
        if not fp.is_file():
            return False
        if fp.stat().st_size == 0:
            return False
        obj = _json.loads(fp.read_text(encoding="utf-8"))
        segs = obj.get("segments") if isinstance(obj, dict) else None
        return isinstance(segs, list) and len(segs) > 0
    except Exception:
        return False

def _safe_move_to_old(src_str, old_dir):
    # Hayato指摘4: old移動は同名衝突時に上書きせず連番で退避
    try:
        Path(old_dir).mkdir(parents=True, exist_ok=True)
        dst = Path(old_dir) / Path(src_str).name
        if not dst.exists():
            _shutil.move(str(src_str), str(dst))
            return str(dst)
        base, suf = dst.stem, dst.suffix
        i = 1
        while True:
            cand = Path(old_dir) / f"{base}_{i}{suf}"
            if not cand.exists():
                _shutil.move(str(src_str), str(cand))
                return str(cand)
            i += 1
    except Exception as e:
        print(f"old移動失敗 {src_str}: {e}")
        return None

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTBASE = DRIVE_OUTDIR
    Path(OUTBASE).mkdir(parents=True, exist_ok=True)
    if not Path(TO_COLAB_DIR).is_dir():
        print(f"入力置き場なし: {TO_COLAB_DIR}")
    else:
        cands = sorted([str(pp) for pp in Path(TO_COLAB_DIR).glob("*") if pp.suffix.lower() in (".m4a", ".mp3", ".wav", ".zip")])
        if not cands:
            print("to_colabに音声なし。m4a/mp3/wav/zipを置くこと")
        for q in cands:
            if q.lower().endswith(".zip"):
                # zipは内包音声名が解凍まで確定しないためスキップ判定しない。常に全件処理する
                FILES.append(q)
                continue
            key = _out_key_for(q)
            if (not REDO) and _is_valid_output(OUTBASE, key):
                SKIPPED.append(q)
                # 書込→old移動の間で落ちた場合の回収: 有効json済みなのに入力が残っていたら次回oldへ退避
                if Path(q).exists():
                    _safe_move_to_old(q, OLD_DIR)
                continue
            FILES.append(q)
        print(f"対象{len(FILES)}件 スキップ{len(SKIPPED)}件")
        print("対象:", FILES)
        print("スキップ:", SKIPPED)
else:
    up = files.upload()
    OUTBASE = "/content/out"
    Path(OUTBASE).mkdir(parents=True, exist_ok=True)
    cands = sorted([k for k in up.keys() if Path(k).suffix.lower() in (".m4a", ".mp3", ".wav", ".zip")])
    for q in cands:
        if q.lower().endswith(".zip"):
            FILES.append(q)
            continue
        key = _out_key_for(q)
        if (not REDO) and _is_valid_output(OUTBASE, key):
            SKIPPED.append(q)
            continue
        FILES.append(q)
    print(f"対象{len(FILES)}件 スキップ{len(SKIPPED)}件")


In [ ]:
# 分離モデル取得 (一次情報: k2-fsa/sherpa-onnx releases。綴りrecongitionは公式通り)
# Drive常駐: MODELS_DIR下に置き、存在すればDL省略。初回のみ取得、2回目以降は数分短縮。存在確認はファイルサイズ>0まで見る
import os as _os3
from pathlib import Path as _Path3
import subprocess as _sp3
_mdir = globals().get("MODELS_DIR", "/content/models")
_seg_int8 = globals().get("SEG_MODEL_INT8", _mdir + "/sherpa-onnx-pyannote-segmentation-3-0/model.int8.onnx")
_seg_fp32 = globals().get("SEG_MODEL_FP32", _mdir + "/sherpa-onnx-pyannote-segmentation-3-0/model.onnx")
_emb = globals().get("EMB_MODEL_PATH", _mdir + "/nemo_en_titanet_small.onnx")
_Path3(_mdir).mkdir(parents=True, exist_ok=True)
def _ok3(p):
    try:
        return _Path3(p).is_file() and _Path3(p).stat().st_size > 0
    except Exception:
        return False
if not (_ok3(_seg_int8) or _ok3(_seg_fp32)):
    _sp3.run(["wget", "-q", "-P", _mdir, "https://github.com/k2-fsa/sherpa-onnx/releases/download/speaker-segmentation-models/sherpa-onnx-pyannote-segmentation-3-0.tar.bz2"], check=True)
    _sp3.run(["tar", "xf", _mdir + "/sherpa-onnx-pyannote-segmentation-3-0.tar.bz2", "-C", _mdir], check=True)
else:
    print("segmentation model exists, skip download")
if not _ok3(_emb):
    _sp3.run(["wget", "-q", "-O", _emb, "https://github.com/k2-fsa/sherpa-onnx/releases/download/speaker-recongition-models/nemo_en_titanet_small.onnx"], check=True)
else:
    print("embedding model exists, skip download")
if _ok3(_seg_int8):
    SEG_MODEL_PATH = _seg_int8
elif _ok3(_seg_fp32):
    SEG_MODEL_PATH = _seg_fp32
elif _ok3("sherpa-onnx-pyannote-segmentation-3-0/model.int8.onnx"):
    SEG_MODEL_PATH = "sherpa-onnx-pyannote-segmentation-3-0/model.int8.onnx"
else:
    SEG_MODEL_PATH = "sherpa-onnx-pyannote-segmentation-3-0/model.onnx"
EMB_MODEL_PATH = _emb if _ok3(_emb) else ("nemo_en_titanet_small.onnx" if _ok3("nemo_en_titanet_small.onnx") else _emb)
print(SEG_MODEL_PATH)
print(EMB_MODEL_PATH)
!ls -lh "$SEG_MODEL_PATH" "$EMB_MODEL_PATH"

In [ ]:
import soundfile as sf
import numpy as np
import torch
import json
import subprocess
import zipfile
import shutil
from pathlib import Path
from transformers import pipeline

def _out_key_for_asr(src_str):
    # セル2と同一規則 (セル単独実行時の退避定義)
    pp = Path(src_str)
    ext = pp.suffix.lower().lstrip(".") or "noext"
    return f"{pp.stem}__{ext}"

def resolve_audios(src):
    # Hayato指摘3: zip内複数音声は全件処理する。1件でも警告ではなく全件列挙する
    pp = Path(src)
    if src.lower().endswith(".zip"):
        stem = pp.stem
        exdir = Path("/content/unzipped") / stem
        with zipfile.ZipFile(src) as z:
            z.extractall(str(exdir))
        auds = sorted([str(q) for q in exdir.rglob("*") if q.suffix.lower() in (".m4a", ".mp3", ".wav")])
        txts = sorted([str(q) for q in exdir.rglob("*.txt")])
        if not auds:
            raise RuntimeError(f"zip内に音声なし: {src}")
        if len(auds) > 1:
            print(f"警告: zip内複数音声 {len(auds)}件を全件処理します: {src}")
            for a in auds:
                print("  -", a)
        else:
            print(f"zip解凍 {stem} Pixel版:", txts[0] if txts else "なし")
        Path(OUTBASE).mkdir(parents=True, exist_ok=True)
        for t in txts:
            dst = Path(OUTBASE) / f"{stem}__{Path(t).stem}_pixel.txt"
            if dst.exists():
                base, suf, i = dst.stem, dst.suffix, 1
                while (Path(OUTBASE) / f"{base}_{i}{suf}").exists():
                    i += 1
                dst = Path(OUTBASE) / f"{base}_{i}{suf}"
            shutil.copy(t, str(dst))
            print("Pixel版を出力先へ複写:", str(dst))
        out = []
        for a in auds:
            inner = Path(a)
            iext = inner.suffix.lower().lstrip(".") or "noext"
            key = f"{stem}__{inner.stem}__{iext}"
            out.append((a, key))
        return out
    return [(src, _out_key_for_asr(src))]

use_gpu = torch.cuda.is_available()
_BATCH = globals().get("BATCH_SIZE", 16)
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo",
               dtype=torch.float16 if use_gpu else torch.float32,
               device="cuda:0" if use_gpu else "cpu")
langmap = {"ja": "japanese", "zh": "chinese"}
gen_kw = {"task": "transcribe"}
if LANGUAGE in langmap:
    gen_kw["language"] = langmap[LANGUAGE]

TASKS = []
for SRC in list(FILES):
    try:
        for _aud, _key in resolve_audios(SRC):
            TASKS.append((str(SRC), _aud, _key))
    except Exception as e:
        print(f"{SRC}: 解凍失敗 {e}")
        FAIL.append((str(SRC), f"unzip:{e}"))
if len(TASKS) != len(FILES):
    print(f"展開: 入力{len(FILES)}件 -> 処理単位{len(TASKS)}件 (zip内複数・stem衝突分離のため)")

WSEGS = {}
for SRC, AUD, key in list(TASKS):
    try:
        wav16 = f"/content/in_16k_{key}.wav"
        subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", AUD, "-ac", "1", "-ar", "16000", wav16], check=True)
        data, sr = sf.read(wav16, dtype="float32", always_2d=True)
        assert sr == 16000
        audio = np.ascontiguousarray(data.mean(axis=1), dtype=np.float32)
        total_s = len(audio) / 16000
        print(f"{key}: 音声 {total_s/60:.1f}分")
        ckpt = Path(OUTBASE) / f"chunks_{key}"
        if REDO and ckpt.is_dir():
            # Hayato指摘1: REDO=True時はチャンクckpを破棄してASR再実行 (看板通り動かす)
            shutil.rmtree(ckpt)
            print(f"{key}: REDOのためチャンクckp破棄")
        ckpt.mkdir(parents=True, exist_ok=True)
        chunk_s, wsegs, ci = CHUNK_MIN * 60, [], 0
        while True:
            base = ci * chunk_s
            if base >= total_s:
                break
            fp = ckpt / f"chunk_{ci:02d}.json"
            if fp.exists() and not REDO:
                part = json.loads(fp.read_text(encoding="utf-8"))
                wsegs += part
                print(f"{key} chunk {ci}: 保存済み {len(part)}件を再利用")
            else:
                lo = max(0, int((base - (OVERLAP_S if ci > 0 else 0)) * 16000))
                hi = min(len(audio), int((base + chunk_s + 30) * 16000))
                res = asr(audio[lo:hi], chunk_length_s=30, stride_length_s=5, batch_size=_BATCH,
                          return_timestamps=True, generate_kwargs=gen_kw)
                part = []
                for c in res["chunks"]:
                    t0, t1 = c.get("timestamp", (None, None))
                    s0, s1 = lo / 16000 + float(t0 or 0.0), lo / 16000 + float(t1 or 0.0)
                    if not (base <= s0 < base + chunk_s):
                        continue  # 自区間開始の文だけ採用。前後チャンクと重複も欠落もなし
                    part.append({"start": s0, "end": s1, "text": str(c["text"]).strip(),
                                 "avg_logprob": 0.0, "no_speech_prob": 0.0})
                fp.write_text(json.dumps(part, ensure_ascii=False), encoding="utf-8")
                wsegs += part
                print(f"{key} chunk {ci}: 起こし {len(part)}件", flush=True)
            ci += 1
        print(f"{key}: 起こし合計 {len(wsegs)}件")
        WSEGS[key] = wsegs
    except Exception as e:
        print(f"{SRC}=>{key}: 起こし失敗 {e}")
        FAIL.append((str(SRC), f"asr:{e}"))


In [ ]:
import time
import json
import re
import shutil
import soundfile as sf
import numpy as np
from pathlib import Path
import sherpa_onnx

# 分離は一括で行うのではない。TASKSで回すが1件ずつ全音声で分離する (チャンク毎だと話者番号が割れるため)
def _ok5(p):
    try:
        return Path(p).is_file() and Path(p).stat().st_size > 0
    except Exception:
        return False
_seg_cands = [globals().get("SEG_MODEL_PATH"), globals().get("SEG_MODEL_INT8"), globals().get("SEG_MODEL_FP32"), "sherpa-onnx-pyannote-segmentation-3-0/model.int8.onnx", "sherpa-onnx-pyannote-segmentation-3-0/model.onnx"]
seg_model = next((c for c in _seg_cands if isinstance(c, str) and _ok5(c)), "sherpa-onnx-pyannote-segmentation-3-0/model.int8.onnx")
if not _ok5(seg_model) and Path(seg_model).exists() is False:
    _alt = "sherpa-onnx-pyannote-segmentation-3-0/model.onnx"
    seg_model = _alt
_emb_cands = [globals().get("EMB_MODEL_PATH"), "nemo_en_titanet_small.onnx"]
emb_model = next((c for c in _emb_cands if isinstance(c, str) and _ok5(c)), globals().get("EMB_MODEL_PATH", "nemo_en_titanet_small.onnx"))
seg_cfg = sherpa_onnx.OfflineSpeakerSegmentationModelConfig(
    pyannote=sherpa_onnx.OfflineSpeakerSegmentationPyannoteModelConfig(model=seg_model))
emb_cfg = sherpa_onnx.SpeakerEmbeddingExtractorConfig(model=emb_model)
clu_cfg = sherpa_onnx.FastClusteringConfig(num_clusters=NUM_SPEAKERS, threshold=CLUSTER_THRESHOLD)
cfg = sherpa_onnx.OfflineSpeakerDiarizationConfig(
    segmentation=seg_cfg, embedding=emb_cfg, clustering=clu_cfg,
    min_duration_on=0.3, min_duration_off=0.5)
dz = sherpa_onnx.OfflineSpeakerDiarization(cfg)

HALL = re.compile("^(ご視聴ありがとうございました|チャンネル登録.+?|高評価.+?|次回もお楽しみに)+")

def ts(x):
    ms = max(0, int(x * 1000))
    return f"{ms//3600000:02d}:{(ms%3600000)//60000:02d}:{(ms%60000)//1000:02d},{ms%1000:03d}"

if "_safe_move_to_old" not in globals():
    def _safe_move_to_old(src_str, old_dir):
        try:
            Path(old_dir).mkdir(parents=True, exist_ok=True)
            dst = Path(old_dir) / Path(src_str).name
            if not dst.exists():
                shutil.move(str(src_str), str(dst))
                return str(dst)
            base, suf, i = dst.stem, dst.suffix, 1
            while True:
                cand = Path(old_dir) / f"{base}_{i}{suf}"
                if not cand.exists():
                    shutil.move(str(src_str), str(cand))
                    return str(cand)
                i += 1
        except Exception as e:
            print(f"old移動失敗 {src_str}: {e}")
            return None

_tasks = globals().get("TASKS")
if not _tasks:
    # セル4未実行の退避: FILESからキーを復元 (zipは展開不可のため対象外)
    def _k(s):
        pp = Path(s)
        return f"{pp.stem}__{pp.suffix.lower().lstrip('.') or 'noext'}"
    _tasks = [(s, s, _k(s)) for s in list(FILES) if not str(s).lower().endswith(".zip")]

for SRC, AUD, key in list(_tasks):
    if key not in WSEGS:
        continue  # 起こし失敗分は飛ばす。FAILに記録済み
    try:
        wsegs = WSEGS[key]
        wav16 = f"/content/in_16k_{key}.wav"
        data, sr = sf.read(wav16, dtype="float32", always_2d=True)
        audio = np.ascontiguousarray(data.mean(axis=1), dtype=np.float32)
        print(f"{key}: 分離開始", flush=True)
        t0 = time.time()
        last = [-1]
        def cb(done, total):
            pct = int(done * 100 / max(1, total))
            if pct >= last[0] + 5:
                last[0] = pct - (pct % 5)
                print(f"分離 {pct}% ({done}/{total})", flush=True)
            return 0
        dsegs_raw = dz.process(audio, callback=cb).sort_by_start_time()
        print(f"{key}: 分離完了 {time.time()-t0:.0f}秒", flush=True)
        dsegs = [{"start": float(s.start), "end": float(s.end), "speaker": f"SPEAKER_{s.speaker:02d}", "confidence": float(getattr(s, "confidence", 0.0))} for s in dsegs_raw]
        print(f"{key}: 分離 {len(dsegs)}区間")
        merged = []
        for w in wsegs:
            text = HALL.sub("", w["text"]).strip()
            if not text:
                continue
            spk, best = "SPEAKER_00", 0.0
            for dd in dsegs:
                ov = max(0.0, min(w["end"], dd["end"]) - max(w["start"], dd["start"]))
                if ov > best:
                    best, spk = ov, dd["speaker"]
            merged.append({"index": len(merged), "start": round(w["start"], 2), "end": round(w["end"], 2),
                           "speaker": spk, "text": text,
                           "avg_logprob": w["avg_logprob"], "no_speech_prob": w["no_speech_prob"]})
        out = Path(OUTBASE)
        out.mkdir(parents=True, exist_ok=True)
        # 原子的書込: tmpに書いてreplace。書込途中で落ちた壊jsonを次回valid扱いしない (Hayato指摘2)
        tmpj = out / f"{key}.json.tmp"
        finj = out / f"{key}.json"
        tmpj.write_text(json.dumps({"segments": merged}, ensure_ascii=False, indent=2), encoding="utf-8")
        tmpj.replace(finj)
        with open(out / f"{key}.srt", "w", encoding="utf-8") as f:
            for m in merged:
                f.write(f"{m['index']+1}\n{ts(m['start'])} --> {ts(m['end'])}\n[{m['speaker']}] {m['text']}\n\n")
        with open(out / f"{key}.txt", "w", encoding="utf-8") as f:
            for m in merged:
                f.write(f"[{ts(m['start'])} {m['speaker']}] {m['text']}\n")
        print("出力:", key, f"{len(merged)}件")
        if USE_DRIVE and Path(SRC).exists():
            # 同一SRC複数key (zip) の2件目以降は既移動済みのためexists確認。衝突時は連番退避
            _safe_move_to_old(str(SRC), OLD_DIR)
            print(f"移動: {SRC} -> {OLD_DIR}/")
        OK.append(f"{SRC}=>{key}")
        if not USE_DRIVE:
            files.download(str(out / f"{key}.json"))
            files.download(str(out / f"{key}.txt"))
    except Exception as e:
        print(f"{SRC}=>{key}: 分離結合失敗 {e}")
        FAIL.append((str(SRC), f"dz:{e}"))

print(f"サマリ 成功{len(OK)}件 失敗{len(FAIL)}件 スキップ{len(SKIPPED)}件")
print("成功:", OK)
print("失敗:", FAIL)
print("スキップ:", SKIPPED)


In [ ]:
# 合図: ブラウザで音を鳴らす (タブを開けて音量を上げること)
import numpy as np
from IPython.display import Audio, display
sr = 22050
t = np.linspace(0, 0.5, int(sr * 0.5), False)
tone = (0.5 * np.sin(2 * np.pi * 880 * t)).astype(np.float32)
for _ in range(3):
    display(Audio(tone, rate=sr, autoplay=True))
print("完了", flush=True)

## 仕上げ (Kai側)

1. 出たtxtをKaiに貼る。Pixel起こしtxtと突合せて誤変換と話者反転を直す。
2. 中国語部分の日本語化もKaiに頼む (whisperのtranslateは英語行きのみ、turboは不可のため)。
3. 話者数が崩れたらNUM_SPEAKERSを固定して分離セルだけ再実行。